# 06 · Data Cleaning

Checks every acquired image (PlantVillage, FieldPlant, Tanzania potato and PlantDoc)
and records which ones to keep in `metadata/clean_manifest.csv`. **Nothing is deleted
from Drive**; the split reads the manifest.

| Rule | Reason recorded |
|---|---|
| Does not open or decode fully | `unreadable` |
| Either side below `config.MIN_IMAGE_SIZE` | `too_small` |
| Same file in two different classes (all copies dropped) | `duplicate_conflicting_labels` |
| Repeat of a file in the same class (first copy kept) | `exact_duplicate` |
| PlantDoc image that is also a training image | `in_training_sources` |
| PlantDoc image that is a near-copy of a training image | `near_duplicate_of_training` |

Near-duplicates inside the training pool (perceptual hash within
`config.NEAR_DUPLICATE_MAX_DISTANCE` bits) are kept but share a `dup_group`, so the
split can keep them in one partition. Runs on a CPU runtime.

In [1]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

Mounted at /content/drive


In [2]:
# Keep the Drive copy of the repository in sync with GitHub.
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

Already up to date.
Refresh index: 100% (90/90), done.
## main...origin/main
 M ml/notebooks/06_image_cleaning.ipynb


In [3]:
%pip install -q -r {ML_DIR}/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 136.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 7.0 MB/s eta 0:00:00


## List every image

In [6]:
expected = {
    "plantvillage": (config.PLANTVILLAGE_DIR, 12631),
    "fieldplant": (config.FIELDPLANT_DIR, 889),
    "tanzania": (config.TANZANIA_DIR, 3000),
    "plantdoc": (config.PLANTDOC_DIR, 1344),
}
for name, (root, want) in expected.items():
    have = sum(1 for p in root.rglob("*") if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"})
    print(f"{name}: {have} of {want} {'OK' if have == want else 'MISSING'}")

print()
for f in sorted(config.METADATA_DIR.iterdir()):
    print(f.name, f.stat().st_size, "bytes")


plantvillage: 9533 of 12631 MISSING
fieldplant: 889 of 889 OK
tanzania: 3000 of 3000 OK
plantdoc: 1344 of 1344 OK

fieldplant_crops.csv 163155 bytes
fieldplant_dropped.csv 528 bytes
plantdoc_images.csv 193749 bytes
tanzania_inventory.csv 5221859 bytes
tanzania_sample.csv 267418 bytes


In [4]:
import pandas as pd

from src.data import cleaning

images = pd.DataFrame(
    cleaning.collect(config.PLANTVILLAGE_DIR, "plantvillage", cleaning.TRAIN_POOL, config.PLANTVILLAGE_CLASSES)
    + cleaning.collect(config.FIELDPLANT_DIR, "fieldplant", cleaning.TRAIN_POOL)
    + cleaning.collect(config.TANZANIA_DIR, "tanzania", cleaning.TRAIN_POOL)
    + cleaning.collect(config.PLANTDOC_DIR / "in_scope", "plantdoc", "ood_in_scope")
    + cleaning.collect(config.PLANTDOC_DIR / "unseen", "plantdoc", "ood_unseen")
)
display(images.groupby(["subset", "source"]).size().rename("images").reset_index())
print("Total:", len(images))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/CropDiseaseDiagnosis/datasets/raw/plantvillage/Potato___Early_blight'

## Check every image

Opens and fully decodes each image, and computes its file hash (exact duplicates) and
perceptual hash (near-duplicates). Reading from Drive is the slow part; expect 10–20 minutes.

In [ ]:
checks = pd.DataFrame(cleaning.check_all(images["path"].tolist()))
checked = pd.concat([images, checks], axis=1)
print("Unreadable:", checked["error"].notna().sum())
display(checked.loc[checked["error"].notna(), ["source", "class", "path", "error"]].head(20))

## Apply the cleaning rules

In [ ]:
manifest = cleaning.clean(checked, config.MIN_IMAGE_SIZE, config.NEAR_DUPLICATE_MAX_DISTANCE)

summary = (
    manifest.assign(reason=manifest["reason"].replace("", "kept"))
    .pivot_table(index=["subset", "source"], columns="reason", values="path", aggfunc="size", fill_value=0)
)
display(summary)

pool = manifest[manifest["dup_group"].notna()]
group_sizes = pool.groupby("dup_group").size()
mixed = pool.groupby("dup_group")["class"].nunique()
print("Kept:", manifest["keep"].sum(), "of", len(manifest))
print("Near-duplicate groups (2+ images):", (group_sizes > 1).sum(),
      "| images in them:", group_sizes[group_sizes > 1].sum(),
      "| largest group:", group_sizes.max())
print("Groups mixing classes:", (mixed > 1).sum())

## Save the manifest

`clean_manifest.csv` has one row per image with its keep flag, reason, hashes and
`dup_group` (paths relative to the raw data folder); `cleaning_summary.csv` holds the
table above for the data report.

In [ ]:
out = manifest.drop(columns="error").copy()
out["path"] = [Path(p).relative_to(config.RAW_DATA_DIR).as_posix() for p in out["path"]]
out.to_csv(config.METADATA_DIR / "clean_manifest.csv", index=False)
summary.to_csv(config.METADATA_DIR / "cleaning_summary.csv")
print("Saved clean_manifest.csv and cleaning_summary.csv to", config.METADATA_DIR)

## Save to GitHub

Commits the changed notebook to a new branch and prints a link to open the pull
request (`main` is protected). Change the branch name if this cell has already been
run once. Save the notebook (Ctrl+S) before running this cell.

In [ ]:
from google.colab import userdata

from src import github_sync

print(github_sync.status(REPO_DIR))
print()
print(github_sync.push(
    REPO_DIR,
    branch="colab/image-cleaning",
    message="Clean the acquired images and save the manifest",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
))